# BigCloneBench Type 1 Dataset Analysis

This notebook reads the artifacts produced by the Type-1 pipeline. It does not rescan the full BCB dump.

In [7]:
from pathlib import Path
import html
import json
import random
import sys

import pandas as pd
from IPython.display import HTML, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "spectral_code").exists() and (candidate / "pipelines").exists():
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from spectral_code.evaluation.bcb_preparation import is_getter_setter
from spectral_code.utils.dataset_paths import bcb_type_dir, output_root_for

CLONE_TYPE = 1
DATA_DIR = bcb_type_dir(CLONE_TYPE)
OUTPUT_ROOT = output_root_for("bcb", CLONE_TYPE)
REPORTS_DIR = OUTPUT_ROOT / "reports"
METADATA_PATH = DATA_DIR / "metadata.json"
PIPELINE_TIMINGS_PATH = OUTPUT_ROOT / "pipeline_timings.json"

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"metadata.json not found in {DATA_DIR}. Run 01_extract_data.py first.")

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
pipeline_timings = json.loads(PIPELINE_TIMINGS_PATH.read_text(encoding="utf-8")) if PIPELINE_TIMINGS_PATH.exists() else {"stages": {}}

print("Project root:", PROJECT_ROOT)
print("Prepared Type-1 data:", DATA_DIR)
print("Output root:", OUTPUT_ROOT)

Project root: c:\Users\koush\PyProjects\spectrals\Spectral-Software
Prepared Type-1 data: C:\Users\koush\PyProjects\spectrals\data\bcb\type1
Output root: C:\Users\koush\PyProjects\spectrals\outputs\bcb\type1


In [8]:
def count_lines(path: Path) -> int:
    with path.open("rb") as f:
        return sum(1 for _ in f)


def unique_code_ids_in_pairs(path: Path) -> int:
    ids = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            left, right, _ = line.rstrip("\n").split("\t")
            ids.add(left)
            ids.add(right)
    return len(ids)


type1_summary = pd.DataFrame(
    [
        {
            "metric": "Type-1 code ids in full BCB",
            "count": metadata.get("full_clone_type_unique_code_ids"),
            "source": "01_extract_data.py metadata",
        },
        {
            "metric": "Type-1 code ids used in extracted data",
            "count": unique_code_ids_in_pairs(DATA_DIR / "train_positives.txt"),
            "source": "train_positives.txt",
        },
        {
            "metric": "Type-1 positive pairs in full BCB",
            "count": metadata.get("full_clone_type_positive_pairs", metadata.get("available_positive_clones")),
            "source": "01_extract_data.py metadata",
        },
        {
            "metric": "Type-1 pairs used in extracted data",
            "count": count_lines(DATA_DIR / "train_positives.txt"),
            "source": "train_positives.txt",
        },
    ]
)
display(type1_summary)

,metric,count,source
0,Type-1 code ids in full BCB,3793,01_extract_data.py metadata
1,Type-1 code ids used in extracted data,3603,train_positives.txt
2,Type-1 positive pairs in full BCB,48062,01_extract_data.py metadata
3,Type-1 pairs used in extracted data,45873,train_positives.txt


In [9]:
sample_counts = {
    "prepared_data_dir": str(DATA_DIR),
    "sampled_functions_written": count_lines(DATA_DIR / "data.jsonl"),
    "sampled_train_pairs": count_lines(DATA_DIR / "train.txt"),
    "sampled_positive_pairs": count_lines(DATA_DIR / "train_positives.txt"),
    "sampled_type_labels": count_lines(DATA_DIR / "type_labels.tsv"),
    "metadata_positive_clones": metadata.get("positive_clones"),
    "metadata_non_clones": metadata.get("non_clones"),
    "metadata_total_pairs": metadata.get("total_pairs"),
    "metadata_written_functions": metadata.get("written_functions"),
    "current_files_getter_setters_removed": metadata.get("current_files_getter_setter_functions_removed", metadata.get("excluded_getter_setter_functions")),
}

display(pd.DataFrame([sample_counts]).T.rename(columns={0: "value"}))

,value
prepared_data_dir,C:\Users\koush\PyProjects\spectrals\data\bcb\t...
sampled_functions_written,56068
sampled_train_pairs,293079
sampled_positive_pairs,45873
sampled_type_labels,45873
metadata_positive_clones,45873
metadata_non_clones,247206
metadata_total_pairs,293079
metadata_written_functions,56068
current_files_getter_setters_removed,3205


In [10]:
def reservoir_sample_by_label(path: Path, label: int, n: int, seed: int) -> list[tuple[str, str, int]]:
    rng = random.Random(seed)
    sample: list[tuple[str, str, int]] = []
    seen = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            left, right, raw_label = line.rstrip("\n").split("\t")
            raw_label = int(raw_label)
            if raw_label != label:
                continue
            row = (left, right, raw_label)
            seen += 1
            if len(sample) < n:
                sample.append(row)
            else:
                replace_at = rng.randrange(seen)
                if replace_at < n:
                    sample[replace_at] = row
    return sample


def load_code_for_ids(path: Path, wanted_ids: set[str]) -> dict[str, str]:
    code_map = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            function_id = str(row["idx"])
            if function_id in wanted_ids:
                code_map[function_id] = row["func"]
                if len(code_map) == len(wanted_ids):
                    break
    return code_map


positive_pairs = reservoir_sample_by_label(DATA_DIR / "train.txt", label=1, n=2, seed=42)
negative_pairs = reservoir_sample_by_label(DATA_DIR / "train.txt", label=0, n=2, seed=43)
wanted_ids = {item for pair in [*positive_pairs, *negative_pairs] for item in pair[:2]}
code_map = load_code_for_ids(DATA_DIR / "data.jsonl", wanted_ids)

positive_pairs = [pair for pair in positive_pairs if pair[0] in code_map and pair[1] in code_map]
negative_pairs = [pair for pair in negative_pairs if pair[0] in code_map and pair[1] in code_map]

print("Loaded code snippets for examples:", len(code_map))
print("Positive examples:", len(positive_pairs))
print("Non-clone examples:", len(negative_pairs))


Loaded code snippets for examples: 8
Positive examples: 2
Non-clone examples: 2


In [11]:
def render_code_pair(pair: tuple[str, str, int], title: str) -> HTML:
    left_id, right_id, label = pair
    left_code = html.escape(code_map.get(left_id, ""))
    right_code = html.escape(code_map.get(right_id, ""))
    label_name = "clone" if label == 1 else "non-clone"
    return HTML(f"""
    <div style="margin: 18px 0 28px 0;">
      <h3 style="margin: 0 0 8px 0; font-family: system-ui, sans-serif;">{html.escape(title)} ({label_name})</h3>
      <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; align-items: start;">
        <div style="border: 1px solid #d0d7de; border-radius: 6px; overflow: hidden;">
          <div style="padding: 6px 10px; background: #f6f8fa; font-family: system-ui, sans-serif; font-size: 13px;">left id: {html.escape(left_id)}</div>
          <pre style="margin: 0; padding: 12px; overflow-x: auto; white-space: pre-wrap; font-size: 12px; line-height: 1.35;">{left_code}</pre>
        </div>
        <div style="border: 1px solid #d0d7de; border-radius: 6px; overflow: hidden;">
          <div style="padding: 6px 10px; background: #f6f8fa; font-family: system-ui, sans-serif; font-size: 13px;">right id: {html.escape(right_id)}</div>
          <pre style="margin: 0; padding: 12px; overflow-x: auto; white-space: pre-wrap; font-size: 12px; line-height: 1.35;">{right_code}</pre>
        </div>
      </div>
    </div>
    """)


rng = random.Random(42)
for idx, pair in enumerate(rng.sample(positive_pairs, min(2, len(positive_pairs))), start=1):
    display(render_code_pair(pair, f"Positive example {idx}"))

for idx, pair in enumerate(rng.sample(negative_pairs, min(2, len(negative_pairs))), start=1):
    display(render_code_pair(pair, f"Non-clone example {idx}"))

In [12]:
from spectral_code.utils.dataset_paths import bcb_type_dir, output_root_for

DATA_DIR = bcb_type_dir(4)
OUTPUT_ROOT = output_root_for("bcb", 4)
PIPELINE_TIMINGS_PATH = OUTPUT_ROOT / "pipeline_timings.json"
METADATA_PATH = DATA_DIR / "metadata.json"
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8")) if METADATA_PATH.exists() else {}


def stage_runtime_row(stage: str) -> dict:
    timings = json.loads(PIPELINE_TIMINGS_PATH.read_text(encoding="utf-8")) if PIPELINE_TIMINGS_PATH.exists() else {"stages": {}}
    record = timings.get("stages", {}).get(stage, {})
    source = str(PIPELINE_TIMINGS_PATH)

    seconds = record.get("seconds")
    minutes = record.get("minutes")
    updated_at_utc = record.get("updated_at_utc")

    if not isinstance(seconds, (int, float)) and stage == "01_extract_data":
        seconds = metadata.get("preparation_time_seconds")
        minutes = seconds / 60 if isinstance(seconds, (int, float)) else None
        source = str(METADATA_PATH)

    if minutes is None and isinstance(seconds, (int, float)):
        minutes = seconds / 60

    return {
        "stage": stage,
        "seconds": round(seconds, 2) if isinstance(seconds, (int, float)) else None,
        "minutes": round(minutes, 2) if isinstance(minutes, (int, float)) else None,
        "updated_at_utc": updated_at_utc,
        "status": "recorded" if isinstance(seconds, (int, float)) else "not recorded yet",
        "source": source,
    }


runtime_df = pd.DataFrame([stage_runtime_row("01_extract_data")])
display(runtime_df)


,stage,seconds,minutes,updated_at_utc,status,source
0,01_extract_data,225.71,3.76,2026-06-28T16:07:19+00:00,recorded,C:\Users\koush\PyProjects\spectrals\outputs\bc...
